In [ ]:
# SwapOS Phase 1 eval on Colab (L4/T4). Port of the Kaggle recipe with the
# hard-won guards: CUDA driver handling, ldd check, Drive caching, purge.
#
# SCOPE = 'smoke': 8B for all roles, 3 tasks (validates the env, ~10 min)
# SCOPE = 'full':  27B Q4 reason+code, 8B Q4 review, full category run
SCOPE = 'smoke'  # change to 'full' for the real eval
CATEGORIES = 'bugfix,feature'  # or 'refactor'
import os, json, subprocess, sys, time, glob, shutil, urllib.request
print('python', sys.version.split()[0], '| scope:', SCOPE, '| categories:', CATEGORIES)
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout[:900])
gpu = subprocess.run(['nvidia-smi', '--query-gpu=memory.total,name', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip().splitlines()
vram = sum(int(line.split()[0].replace(',', '')) for line in gpu if line) if gpu else 0
print('GPU count:', len(gpu), '| total VRAM MiB:', vram)
if SCOPE == 'full':
    assert vram >= 20000, 'full scope needs >=20GB VRAM (L4/A100) — T4 15GB cannot hold 27B Q4; change runtime type'
elif vram < 8000:
    print('WARNING: <8GB VRAM — set Runtime -> Change runtime type -> Hardware accelerator: GPU')
print(subprocess.run(['df', '-h', '/content'], capture_output=True, text=True).stdout)


In [ ]:
# Optional Drive mount for caching (built llama.cpp + GGUFs). Interactive
# runs: a popup appears. Headless (colab-cli): pre-auth with gcloud.
DRIVE = None
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive/swapos-cache'
    os.makedirs(DRIVE, exist_ok=True)
    print('drive mounted at', DRIVE)
except Exception as e:
    print('drive unavailable (caching off):', str(e)[:100])


In [ ]:
os.chdir('/content')
r = subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/mattdani21/ModelSwapper.git'], capture_output=True, text=True)
print(r.stdout[-200:], r.stderr[-200:])
os.chdir('/content/ModelSwapper')
print(subprocess.run(['git', 'log', '-1', '--format=%h %ci'], capture_output=True, text=True).stdout)
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'pytest'], capture_output=True, text=True)
print('pytest install rc:', r.returncode)


In [ ]:
# Models. Colab's own network downloads from HF (not the user's). Cached in
# Drive so re-runs skip downloads. Full set: 27B Q4 (16.8GB, lmstudio build)
# for reason+code, 8B Q4 for the critic. Smoke: 8B everywhere.
os.makedirs('/content/models', exist_ok=True)
HF = {
    'Qwen3.8-27B-Q4_K_M.gguf': 'https://huggingface.co/lmstudio-community/Qwen3.8-27B-GGUF/resolve/main/Qwen3.8-27B-Q4_K_M.gguf',
    'Qwen3-8B-Q4_K_M.gguf': 'https://huggingface.co/Qwen/Qwen3-8B-GGUF/resolve/main/Qwen3-8B-Q4_K_M.gguf',
}
# Exact-size guards: truncated downloads are the silent killer (instant
# 0/50 = load crash, often a corrupt GGUF). 98% of expected size required.
EXPECTED = {'Qwen3.8-27B-Q4_K_M.gguf': 16.7e9, 'Qwen3-8B-Q4_K_M.gguf': 5.0e9}
def check_size(path, fname):
    sz = os.path.getsize(path)
    assert sz > 0.98 * EXPECTED[fname], f'{fname} incomplete: {sz / 1e9:.2f}GB (expected ~{EXPECTED[fname] / 1e9:.1f}GB)'
    return path
if SCOPE == 'smoke':
    ROLES = {'reason': 'Qwen3-8B-Q4_K_M.gguf', 'code': 'Qwen3-8B-Q4_K_M.gguf', 'review': 'Qwen3-8B-Q4_K_M.gguf'}
    LIMIT = 3
else:
    ROLES = {'reason': 'Qwen3.8-27B-Q4_K_M.gguf', 'code': 'Qwen3.8-27B-Q4_K_M.gguf', 'review': 'Qwen3-8B-Q4_K_M.gguf'}
    LIMIT = 0
MODEL_PATHS = {}
for role, fname in ROLES.items():
    dst = '/content/models/' + fname
    if DRIVE and os.path.exists(DRIVE + '/' + fname) and os.path.getsize(DRIVE + '/' + fname) > 1e9:
        shutil.copy(DRIVE + '/' + fname, dst)
        print('drive cache:', fname)
    elif not (os.path.exists(dst) and os.path.getsize(dst) > 1e9):
        print('downloading', fname)
        subprocess.run(['wget', '-q', '-O', dst, HF[fname]], check=True, timeout=3600)
        if DRIVE:
            shutil.copy(dst, DRIVE + '/' + fname)
            print('cached to drive:', fname)
    MODEL_PATHS[role] = check_size(dst, fname)
    print(role, '->', round(os.path.getsize(dst) / 1e9, 2), 'GB')
print(MODEL_PATHS)


In [ ]:
# llama.cpp with CUDA. Colab images have a standard driver layout, but be
# defensive: locate libcuda, symlink into the toolkit if needed, then an ldd
# 'not found' gate. Drive-cached so re-runs skip the ~15 min build.
os.chdir('/content')  # cell 3 left CWD at /content/ModelSwapper (run-1 fix)
if DRIVE and os.path.exists(DRIVE + '/llama-bin/llama-server'):
    shutil.copytree(DRIVE + '/llama-bin', '/content/llama-bin', dirs_exist_ok=True)
    print('llama-server from drive cache')
else:
    r = subprocess.run(['bash', '-c', 'ldconfig -p | grep -c libcuda.so'], capture_output=True, text=True)
    if r.stdout.strip() == '0':
        for cand in ['/usr/lib/x86_64-linux-gnu/libcuda.so.1', '/usr/local/cuda/lib64/stubs/libcuda.so']:
            if os.path.exists(cand):
                subprocess.run(['bash', '-c', 'ln -sf ' + cand + ' /usr/local/cuda/lib64/libcuda.so'], capture_output=True, text=True)
                print('symlinked', cand)
                break
    subprocess.run(['rm', '-rf', '/content/llama.cpp'], capture_output=True, text=True)
    for attempt in range(4):
        r = subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/ggml-org/llama.cpp.git'], capture_output=True, text=True)
        if r.returncode == 0:
            break
        print(f'clone attempt {attempt + 1} failed: {(r.stderr or "")[-300:]}')
        time.sleep(10)
    assert os.path.isdir('/content/llama.cpp/.git'), 'llama.cpp clone failed after retries'
    r = subprocess.run(['cmake', '-B', '/content/llama.cpp/build', '-S', '/content/llama.cpp',
                        '-DGGML_CUDA=ON', '-DCMAKE_BUILD_TYPE=Release'],
                       capture_output=True, text=True, timeout=900)
    print('cmake configure rc:', r.returncode)
    if r.returncode != 0:
        print(r.stdout[-1200:]); print(r.stderr[-1200:])
    nproc = subprocess.run(['nproc'], capture_output=True, text=True).stdout.strip() or '4'
    r = subprocess.run(['cmake', '--build', '/content/llama.cpp/build', '--config', 'Release',
                        '-j', nproc, '--target', 'llama-server'], capture_output=True, text=True, timeout=3600)
    print('cmake build rc:', r.returncode)
    if r.returncode != 0:
        print(r.stdout[-1200:]); print(r.stderr[-1200:])
    shutil.copytree('/content/llama.cpp/build/bin', '/content/llama-bin', dirs_exist_ok=True)
    subprocess.run(['rm', '-rf', '/content/llama.cpp'])
    if DRIVE:
        shutil.copytree('/content/llama-bin', DRIVE + '/llama-bin', dirs_exist_ok=True)
        print('cached llama-bin to drive')
os.environ['LLAMA_SERVER'] = '/content/llama-bin/llama-server'
os.environ['PATH'] = '/content/llama-bin:' + os.environ['PATH']
os.environ['LD_LIBRARY_PATH'] = '/content/llama-bin:' + os.environ.get('LD_LIBRARY_PATH', '')
r = subprocess.run(['bash', '-c', 'ldd /content/llama-bin/llama-server | grep "not found" || true'], capture_output=True, text=True)
assert not r.stdout.strip(), 'missing shared libs: ' + r.stdout
print('llama-server ready (libs ok)')


In [ ]:
os.chdir('/content/ModelSwapper')
env = dict(os.environ)
env['LLAMA_CONTEXT'] = '4096'
env['LLAMA_NGPU'] = '99'
cmd = [sys.executable, 'pipeline/run_pipeline.py',
       '--models-json', json.dumps(MODEL_PATHS),
       '--out', '/content/pipeline-results.json',
       '--capsule-dir', '/content/capsules',
       '--categories', CATEGORIES,
       '--limit', str(LIMIT),
       '--port-base', '8950',
       '--max-iterations', '3',
       '--max-tokens', '2048']
print('running pipeline eval...')
t0 = time.time()
try:
    r = subprocess.run(cmd, env=env, capture_output=True, text=True, timeout=8 * 3600)
    print('pipeline rc:', r.returncode, '| wall:', round((time.time() - t0) / 60, 1), 'min')
    print((r.stdout or '')[-2500:])
    print((r.stderr or '')[-1000:])
except subprocess.TimeoutExpired:
    print('PIPELINE TIMEOUT after', round((time.time() - t0) / 60, 1), 'min — partial results kept')
shutil.rmtree('/content/models', ignore_errors=True)
print('models purged')


In [ ]:
res_path = '/content/pipeline-results.json'
if os.path.exists(res_path):
    d = json.load(open(res_path))
    print('PASS RATE:', d.get('pass_rate'), '|', d.get('tasks_passed'), '/', d.get('tasks_total'))
    print('per_category:', d.get('per_category'))
    print('mean_wall_clock_s:', d.get('mean_wall_clock_s'))
    print('mean_load_s:', d.get('mean_load_s'), '| mean_evict_s:', d.get('mean_evict_s'))
    shutil.make_archive('/content/results', 'zip', '/content', 'pipeline-results.json')
    shutil.make_archive('/content/capsules', 'zip', '/content', 'capsules')
    if DRIVE:
        for f in ['pipeline-results.json', 'results.zip', 'capsules.zip']:
            src = '/content/' + f
            if os.path.exists(src):
                shutil.copy(src, DRIVE + '/' + f)
                print('saved to drive:', f)
else:
    print('NO RESULTS FILE')
print('done')
